## Note:
This script processes the movie stimulus and extracts lexical surprisal and word frequency features using *Despicable Me* (DM) as an example.  
The same procedure applies to *The Present* (TP).

## 1. convert videos to mono audio

In [ ]:
%%bash
ffmpeg -i stimuli_movieDM.mp4 -vn -acodec pcm_s16le -ar 44100 -ac 1 stimuli_movieDM_audio.wav

## 2. transcribe audio with Whisper and then manually double-check the content

In [ ]:
import whisper

model = whisper.load_model("base")

result = model.transcribe("stimuli_movieDM.mp4")

print(result['text'])

with open("movieDM_transcription_alloneline.txt", "w", encoding="utf-8") as f:
    f.write(result['text'])

## 3. manually annotate the word onset and offset in Praat

(https://www.fon.hum.uva.nl/praat/)

input: stimuli_movieDM_audio.wav

output after annotation: stimuli_movieDM_audio.TextGrid

## 4. extract word onset and offset from .TextGrid files

In [ ]:
import re
import csv

with open('stimuli_movieDM_audio.TextGrid', 'r') as file:
    textgrid_data = file.read()

# Regular expression to match each interval's xmin, xmax, and text
pattern = re.compile(r'xmin = ([\d\.]+)\s+xmax = ([\d\.]+)\s+text = "(.*?)"')

# Extract all intervals with their xmin, xmax, and text
intervals = pattern.findall(textgrid_data)

# Filter out intervals where the text is empty
filtered_intervals = [(float(xmin), float(xmax), text) for xmin, xmax, text in intervals if text]

# Define the CSV file name
csv_filename = 'movieDM_words.csv'

# Write the filtered intervals to a CSV file
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    # Write the header
    writer.writerow(['word', 'onset', 'offset'])
    
    # Write the word, onset, and offset for each interval
    for onset, offset, word in filtered_intervals:
        writer.writerow([word, onset, offset])

print(f"Data successfully saved to {csv_filename}")


## 5. obtain linguistic regressors

### a. lexical surprisal

will need API key to run the following code

In [ ]:
import pandas as pd
import math
import numpy as np
import openai

df = pd.read_csv("movieDM_words.csv")  
used_df = df[["word","onset"]]
print("Original length",len(used_df))

# Concatenate all words into one input string
input_string = " ".join(used_df['word'].astype(str).tolist())
print("Input string:", input_string)

# Function to check if token starts a new word
def is_new_word(token):
    return token.startswith(" ")

# Set API key
openai.api_key = "YOUR_OPENAI_API_KEY"

# Call OpenAI API to get token-level logprobs
response = openai.Completion.create(
    engine="davinci-002",
    prompt=input_string,
    echo=True,
    max_tokens=0,
    logprobs=0
)

# Extract token-level outputs
output = response.choices[0].logprobs
token_text = output.tokens
token_logprobs = output.token_logprobs

# Process to get word-level surprisals
word_surprisals = []
current_word_surprisal = 0.0
current_word = ""

for token, logprob in zip(token_text, token_logprobs):
    if token == token_text[0]: #first word (if only 1 token)
        current_word = token
        current_word_surprisal = -math.log2(np.exp(logprob)) if logprob is not None else 0.0
    elif is_new_word(token): # one word with only 1 token
        word_surprisals.append((current_word, current_word_surprisal))
        current_word = token
        current_word_surprisal = -math.log2(np.exp(logprob)) if logprob is not None else 0.0
    else:
        if logprob is not None: 
            current_word_surprisal += -math.log2(np.exp(logprob)) 
            current_word += token  # Append subword tokens to current word

# Append the final word
if current_word:
    word_surprisals.append((current_word, current_word_surprisal))

# Strip leading spaces in the GPT tokens to align with your CSV 'word' column
gpt_words = [word.strip() for word, _ in word_surprisals]
gpt_surprisals = [surprisal for _, surprisal in word_surprisals]

# Sanity check: ensure lengths match before merging
if len(df) != len(gpt_surprisals):
    raise ValueError(f"Mismatch: CSV has {len(df)} words, but got {len(gpt_surprisals)} surprisal values.")

# Add surprisal column to the original DataFrame
used_df['GPT3_word'] = gpt_words
used_df['GPT3_surprisal'] = gpt_surprisals

# Save to a new CSV file
used_df.to_csv("movieDM_words_GPT3_surprisal.csv", index=False) 

### b. word frequency count

we use the SUBTLEX_US word frequencies from: https://osf.io/djpqz/metadata/osf

ref: Brysbaert, M., & New, B. (2009). Moving beyond Kučera and Francis: A critical evaluation of current word frequency norms and the introduction of a new and improved word frequency measure for American English. Behavior research methods, 41(4), 977-990.

In [ ]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize

# Load SUBTLEX-US Lg10WF frequency data
subtlex_df = pd.read_csv("SUBTLEX-US_freq.csv")
subtlex_df = subtlex_df[["Word", "Lg10WF"]]
subtlex_df = subtlex_df.dropna()  # Drop rows with missing frequencies
subtlex_df["Word"] = subtlex_df["Word"].str.lower()  # Normalize casing
freq_dict = dict(zip(subtlex_df["Word"], subtlex_df["Lg10WF"]))

# load input file
df = pd.read_csv("movieDM_words.csv")
used_df = df[["word", "onset"]]

# Function to handle unknown or compound words
def get_log10_frequency(word):
    word = str(word).lower()
    if word in freq_dict:
        return freq_dict[word]
    
    # Tokenize and try subwords
    tokens = word_tokenize(word)
    print(f"Trying to split... {word} -> {' + '.join(tokens)}")
    # use average of the lexical frequency for compound words
    token_freqs = [freq_dict[token] for token in tokens if token in freq_dict]
    if token_freqs:
        return np.mean(token_freqs)
    else:
        return 0  # Default frequency for completely unknown items

# Apply function to compute lexical frequency
used_df["Lg10WF"] = used_df["word"].apply(get_log10_frequency)

# Save updated CSV
output_file = "movieDM_words_subtlex_Lg10WF.csv"
used_df.to_csv(output_file, index=False)

print(f"Saved with log10 frequencies: {output_file}")